### Setup

In [1]:
import time
import psutil
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
model_name = "distilgpt2"  # small + fast

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token 

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()
print(model_name)

distilgpt2


### Utility Functions (Latency + Memory)

In [3]:
def run_inference(prompt, max_new_tokens=50, generate=True):
    process = psutil.Process()
    mem_before = process.memory_info().rss / 1e6
    start = time.time()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    with torch.no_grad():
        if generate:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id
            )
            text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        else:
            # Forward pass only (no generation)
            outputs = model(**inputs)
            text = "[No generation – forward pass only]"

    latency = time.time() - start
    mem_after = process.memory_info().rss / 1e6

    return {
        "latency_sec": round(latency, 3),
        "memory_mb": round(mem_after - mem_before, 2),
        "output": text
    }

### Example Inputs

In [4]:
short_text = "Explain what retrieval-augmented generation is."

long_text = """
Large language models are powerful but face limitations in context length.
Retrieval-augmented generation helps by fetching relevant documents at inference time.
This allows the model to reason over larger corpora efficiently.
Smart inference systems decide whether retrieval is needed based on input size and intent.
""" * 20  # artificially long

### Full-Context Inference

In [5]:
print("FULL CONTEXT – SHORT INPUT")
run_inference(short_text, generate=True)

FULL CONTEXT – SHORT INPUT


{'latency_sec': 2.757,
 'memory_mb': 330.96,
 'output': 'Explain what retrieval-augmented generation is.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}

In [6]:
print("FULL CONTEXT – LONG INPUT")
run_inference(long_text, generate=False)

FULL CONTEXT – LONG INPUT


{'latency_sec': 3.651,
 'memory_mb': 319.38,
 'output': '[No generation – forward pass only]'}

### Simple Retrieval-Based Approach (Toy Example)

In [7]:
def retrieve_relevant_chunk(query, document, chunk_size=200):
    chunks = [document[i:i+chunk_size] for i in range(0, len(document), chunk_size)]
    scores = [(chunk.lower().count(query.lower()), chunk) for chunk in chunks]
    return max(scores, key=lambda x: x[0])[1]

In [8]:
query = "retrieval"

retrieved_chunk = retrieve_relevant_chunk(query, long_text)
retrieved_chunk[:300]

' systems decide whether retrieval is needed based on input size and intent.\n\nLarge language models are powerful but face limitations in context length.\nRetrieval-augmented generation helps by fetching'

In [9]:
print("RETRIEVAL-BASED INFERENCE")
retrieval_result = run_inference(
    f"Context:\n{retrieved_chunk}\n\nQuestion: Explain retrieval-augmented generation."
)
retrieval_result

RETRIEVAL-BASED INFERENCE


{'latency_sec': 2.751,
 'memory_mb': 0.0,
 'output': 'Context:\n systems decide whether retrieval is needed based on input size and intent.\n\nLarge language models are powerful but face limitations in context length.\nRetrieval-augmented generation helps by fetching\n\nQuestion: Explain retrieval-augmented generation.\nThe problem is that the retrieval-augmented generation is not a good idea.\nThe problem is that the retrieval-augmented generation is not a good idea.\nThe problem is that the retrieval-augmented generation is not'}

### Smart Inference Router (Key Teaching Moment)

In [10]:
def smart_inference(prompt, full_text=None, token_threshold=300):
    token_count = len(tokenizer(prompt)["input_ids"])

    if token_count < token_threshold:
        strategy = "FULL CONTEXT"
        result = run_inference(prompt)
    else:
        strategy = "RETRIEVAL"
        retrieved = retrieve_relevant_chunk(prompt, full_text)
        result = run_inference(
            f"Context:\n{retrieved}\n\nQuestion: {prompt}"
        )

    return strategy, result

In [11]:
strategy, result = smart_inference(
    "Explain retrieval-augmented generation",
    full_text=long_text
)

strategy, result

('FULL CONTEXT',
 {'latency_sec': 2.459,
  'memory_mb': 0.0,
  'output': 'Explain retrieval-augmented generation of the data.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'})